# Import libraries


In [1]:
import pandas as pd
import psycopg2
import uuid
from datetime import datetime
from tqdm import tqdm
import os
import logging
from sqlalchemy import create_engine, Table, MetaData, Column, Integer, Float, Boolean, String, DateTime, ForeignKey, select, update, bindparam, text
from sqlalchemy.orm import sessionmaker, relationship, declarative_base, clear_mappers
from dotenv import load_dotenv
import matplotlib as plt
from textwrap import dedent
import json

load_dotenv()

True

# Get dotenv variables

In [2]:
# Database information
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_db = os.getenv("POSTGRES_DB")

# Data path
data_path = os.getenv("DATA_PATH")

# Connect to database


## Create connection to database

In [3]:
try:
    conn = psycopg2.connect(
        database=postgres_db,
        user=postgres_user,
        host=postgres_host,
        password=postgres_password,
        port=postgres_port,
    )
    print("Opened database successfully")
except Exception as e:
    print(f"Connection failed: {e}")

Opened database successfully


In [4]:
# Create engine
engine = create_engine(
    f"postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}"
)

# Create a session
session = sessionmaker(bind=engine)()

# Create a MetaData instance
metadata = MetaData()

# Reflect the table
company_table = Table("company", metadata, autoload_with=engine)
news_table = Table("news", metadata, autoload_with=engine)
sentiment_table = Table("sentiment", metadata, autoload_with=engine)
statement_table = Table("financialstatement", metadata, autoload_with=engine)

# Define table

In [5]:
Base = declarative_base()
clear_mappers()

## Define company table

In [6]:
class Company(Base):
    __tablename__ = 'company'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    createdAt = Column(DateTime)
    updatedAt = Column(DateTime)
    name = Column(String)
    symbol = Column(String)
    description = Column(String)
    logo = Column(String)
    status = Column(Integer)
    exchange = Column(String)
    circulatingStockVolume = Column(Integer)
    listedStockVolume = Column(Integer)
    marketCap = Column(Integer)
    charterCapital = Column(Integer)
    order = Column(Integer)

    news_relation = relationship("News", back_populates="company_relation")
    stock_relation = relationship("Stock", back_populates="company_relation")
    statement_relation = relationship("Statement", back_populates="company_relation")
    report_relation = relationship("Report", back_populates="company_relation")


## Define statement table

In [7]:
class Statement(Base):
    __tablename__ = 'financialstatement'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    createdAt = Column(DateTime)
    updatedAt = Column(DateTime)
    PE = Column(Float)
    ROA = Column(Float)
    ROE = Column(Float)
    EPS = Column(Float)
    BVPS = Column(Float)
    DAR = Column(Float)
    GOS = Column(Float)
    year = Column(Integer)
    order = Column(Integer) 
    company = Column(String, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="statement_relation")

## Define news table

In [8]:
class News(Base):
    __tablename__ = 'news'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    createdAt = Column(DateTime)
    updatedAt = Column(DateTime)
    title = Column(String)
    rawContent = Column(String)
    cleanContent = Column(String)
    url = Column(String)
    banner = Column(String)
    source = Column(String)
    publishedAt = Column(DateTime)
    order = Column(Integer)
    company = Column(String, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="news_relation")

## Define stock table

In [9]:
class Stock(Base):
    __tablename__ = 'stock'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    createdAt = Column(DateTime)
    updatedAt = Column(DateTime)
    openingPrice = Column(Float)
    closingPrice = Column(Float)
    ceilingPrice = Column(Float)
    floorPrice = Column(Float)
    highestPrice = Column(Float)
    lowestPrice = Column(Float)
    adjustedPrice = Column(Float)
    change = Column(String)
    tradedValue = Column(Integer)
    tradedVolume = Column(Integer)
    agreedValue = Column(Integer)
    agreedVolume = Column(Integer)
    date = Column(DateTime)
    order = Column(Integer)
    company = Column(String, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="stock_relation")

## Define report table

In [10]:
class Report(Base):
    __tablename__ = 'report'
    __table_args__ = {'extend_existing': True}
    id = Column(String, primary_key=True)
    createdAt = Column(DateTime)
    updatedAt = Column(DateTime)
    year = Column(Integer)
    quarter = Column(Integer)
    month = Column(Boolean)
    detail = Column(String)
    mScore1 = Column(Float)
    mScore2 = Column(Float)
    fScore1 = Column(Float)
    zScore1 = Column(Float)
    zScore2 = Column(Float)
    zScore3 = Column(Float)
    order = Column(Integer)
    company = Column(String, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="report_relation")

# Most used function

## Generate UUID function


In [11]:
def generate_uuid():
    id = str(uuid.uuid4())
    created_at = datetime.now()
    updated_at = created_at
    return id, created_at, updated_at

## Get exchange name


In [12]:
def get_exchange_name(exchangeId):
    if exchangeId == 1:
        return "HOSE"
    elif exchangeId == 2:
        return "HNX"
    elif exchangeId == 8:
        return "OTC"
    elif exchangeId == 9:
        return "UPCOM"

In [13]:
def get_company(filename, raw = False):
    df = pd.read_csv(f"{data_path}/raw/{filename}.csv") if raw else pd.read_csv(f"{data_path}/beautiful/{filename}.csv")
    hose_1 = 0
    hnx_2 = 0
    otc_8 = 0
    upcom_9 = 0
    
    exchange_counts = df['ExchangeId'].value_counts()
    
    hose_1 = exchange_counts.get(1, 0)
    hnx_2 = exchange_counts.get(2, 0)
    otc_8 = exchange_counts.get(8, 0)
    upcom_9 = exchange_counts.get(9, 0)

    print(f"HOSE: {hose_1}, HNX: {hnx_2}, OTC: {otc_8}, UPCOM: {upcom_9}")
    print(f"Total: {sum([hose_1, hnx_2, upcom_9])}")

## Read csv file

In [14]:
def read_data(filename, raw = False):
    data = pd.read_csv(f"{data_path}/raw/{filename}.csv") if raw else pd.read_csv(f"{data_path}/beautiful/{filename}.csv")
    return data

## Convert date string to timestamp

In [15]:
def convert_to_timestamp(date_str):
    # Parse the date string
    # date_obj = datetime.strptime(date_str, "%d-%m-%Y")
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    # Convert to the desired timestamp format
    timestamp_str = date_obj.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    return timestamp_str

## Calculate ceiling & floor price

In [16]:
def calculate_ceiling_floor_price(exchange, prev_close_price):
    if exchange == "HOSE":
        ceiling_price = prev_close_price * 1.07
        floor_price = prev_close_price * 0.93
    elif exchange == "HNX":
        ceiling_price = prev_close_price * 1.10
        floor_price = prev_close_price * 0.90
    elif exchange == "UPCOM":
        ceiling_price = prev_close_price * 1.15
        floor_price = prev_close_price * 0.85
    else:
        ceiling_price = 0
        floor_price = 0
    return ceiling_price, floor_price

# Import data

## Company

### Compare csv file with database


In [20]:
def compare_data(filename, raw = False):
    results = session.query(Company.symbol).all()
    list_company_db = [result[0] for result in results]

    new_companies = pd.read_csv(f"{data_path}/raw/{filename}.csv") if raw else pd.read_csv(f"{data_path}/beautiful/{filename}.csv")
    list_company_new = new_companies["Symbol"].tolist()

    diff = list(set(list_company_new) - set(list_company_db))

    print(dedent(f"""
        Amount of companies in database: {len(list_company_db)}
        Amount of new/different companies: {len(list_company_new)}
        Amount of companies in new list: {len(diff)}
    """))

In [19]:
compare_data("company/company")


Amount of companies in database: 1565
Amount of new/different companies: 1652
Amount of companies in new list: 87



### Insert company

In [9]:
def insert_company(filename):
    company = pd.read_csv(f"{data_path}/{filename}.csv")
    for _, row in company.iterrows():
        cursor = conn.cursor()
        if row["Name"]  == "N/A" or row["Symbol"] == "N/A":
            continue
        try:
            # Get company information
            name = row["Name"]
            symbol = row["Symbol"]
            category = row["Category"]

            # Check if industry exists
            industry_rows = company[company["Symbol"] == symbol]["Category Name"]
            if not industry_rows.empty:
                industry = industry_rows.values[0]
            else:
                industry = "N/A"

            description = row["Description"]
            logo = row["Logo"]
            exchangeId = row["ExchangeId"]

            if exchangeId == 8:
                continue

            exchange = get_exchange_name(exchangeId)

            # Check if company exists
            cursor.execute(
                """SELECT id, name, symbol FROM company WHERE symbol = %s""", (symbol,)
            )
            company_record = cursor.fetchone()
            # If company exists, skip
            if company_record:
                continue

            # Check if category is empty
            if pd.notna(category) and pd.notnull(category):
                # Check if category exists
                cursor.execute(
                    """SELECT id, name FROM category WHERE name = %s""", (category,)
                )
                category_record = cursor.fetchone()
                # If category does not exist, create it
                if category_record:
                    category_id = category_record[0]
                    category = category_record[1]
                else:
                    category_id, created_at, updated_at = generate_uuid(category)
                    cursor.execute(
                        """INSERT INTO category (id, name, "createdAt", "updatedAt", "order")
                        VALUES (%s, %s, %s, %s, %s)""",
                        (category_id, category, created_at, updated_at, 0),
                    )
                    conn.commit()
                print(f"Category: {category}")
            else:
                category_id = None

            # Check if industry is in database
            if (
                industry != "N/A"
                and industry is not None
                and pd.notna(industry)
                and pd.notnull(industry)
            ):
                cursor.execute(
                    """SELECT id, name FROM industry WHERE name = %s""", (industry,)
                )
                industry_record = cursor.fetchone()
                # If industry does not exist, create it
                if industry_record:
                    industry_id = industry_record[0]
                    industry = industry_record[1]
                else:
                    industry_id, created_at, updated_at = generate_uuid(category)
                    cursor.execute(
                        """INSERT INTO industry (id, name, "createdAt", "updatedAt", category, "order")
                        VALUES (%s, %s, %s, %s, %s, %s)""",
                        (industry_id, industry, created_at, updated_at, category_id, 0),
                    )
                    conn.commit()
                print(f"Industry: {industry}")
            else:
                industry_id = None

            company_id, created_at, updated_at = generate_uuid(category)
            # Insert company record
            cursor.execute(
                """INSERT INTO company (id, name, symbol, description, logo, status, "createdAt", "updatedAt", exchange, industry, "order")
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
                (
                    company_id,
                    name,
                    symbol,
                    description,
                    logo,
                    1,
                    created_at,
                    updated_at,
                    exchange,
                    industry_id,
                    0,
                ),
            )
            conn.commit()
            print(f"Company: {name}")
        except Exception as e:
            conn.rollback()
            print(f"Error: {e}")
            break
        finally:
            cursor.close()

### Get company id

In [17]:
def get_company_id():
    results = session.query(Company.id, Company.symbol).all()
    company_id = {result[1]: result[0] for result in results}
    return company_id

## News


### Get news from CafeF


### Update news relation

### Count news for top company

In [ ]:
metadata = MetaData()

news_table = Table('news', metadata, autoload_with=engine)
stmt = select(news_table.c.company).where(news_table.c.company.isnot(None))
news_df = pd.read_sql(stmt, engine)

company_table = Table('company', metadata, autoload_with=engine)
stmt = select(company_table.c.id, company_table.c.symbol).where(company_table.c.id.isnot(None))
company_df = pd.read_sql(stmt, engine)

company_dict = {
    row["id"]: row["symbol"] for _, row in company_df.iterrows()
}

# count company in news
company_count = news_df["company"].value_counts()

# map company count to company dict
company_count_dict = {
    company_dict[company_id]: count
    for company_id, count in company_count.items()
}

output = pd.DataFrame(company_count_dict.items(), columns=["symbol", "count"])
output[output["symbol"] == "VCB"]

,symbol,count
49,VCB,299


In [ ]:
company_query = text(
    """
    SELECT id, symbol FROM company
    """
)

# Lấy danh sách các công ty từ database
company_df = pd.read_sql(company_query, engine)

# Lưu thành dictionary
company_dict = {
    row["symbol"]: row["id"] for _, row in company_df.iterrows()
}

In [ ]:
company_dict["AAA"]

'1df9a54d-52bd-4322-a2aa-0853b8145997'

In [ ]:
news = pd.read_csv(f"{data_path}/cafef/news.csv")
news.notnull().sum()

Date       100604
Symbols    100688
URL        100688
Title      100604
Content    100604
dtype: int64

In [ ]:
# Define metadata and table object
metadata = MetaData()
news_table = Table('news', metadata, autoload_with=engine)

# Define the update statement with non-conflicting parameter names
stmt = update(news_table).where(news_table.c.title == bindparam('b_title')).values(company=bindparam('b_company'))

data = []
batch_size = 1000
count = 0

for _, row in tqdm(news.iterrows()):
    if pd.isna(row["Symbols"]) or pd.isnull(row["Symbols"]) or pd.isna(row["Title"]) or pd.isnull(row["Title"]):
        continue

    symbol = row["Symbols"]
    title = row["Title"]

    # Lấy id của công ty từ dictionary
    company_id = company_dict.get(symbol)

    if company_id is None:
        continue

    data.append({"b_title": title, "b_company": company_id})
    count += 1

    if count % batch_size == 0:
        with engine.connect() as conn:
            conn.execute(stmt, data)
            conn.commit()
            data = []

if data:
    try:
        with engine.connect() as conn:
            conn.execute(stmt, data)
            conn.commit()
            print("Remaining records updated.")
    except Exception as e:
        print(f"Error occurred: {e}")


100688it [47:48, 35.11it/s]


Remaining records updated.


In [23]:
news = pd.read_csv(f"{data_path}/cafef/news.csv")
news.notnull().sum()
news.isnull().sum()

Date       84
Symbols     0
URL         0
Title      84
Content    84
dtype: int64

In [11]:
null_rows = news[news.isnull().any(axis=1)]
# null_rows.to_csv(f"{data_path}/news_cafef_null.csv", index=False)
null_rows["Symbols"].unique()

array(['BST', 'DIG', 'HDC', 'KDC', 'LHG', 'PVT', 'THG', 'THM', 'THN',
       'THP', 'THS', 'THT', 'TIN'], dtype=object)

In [12]:
banner = pd.read_csv(f"{data_path}/cafef/banner.csv")
banner.notnull().sum()

Symbols    111733
Title      111733
imgUrl     111733
URL        111733
dtype: int64

In [ ]:
for index, row in tqdm(news.iterrows(), total=news.shape[0]):
    cursor = conn.cursor()
    if pd.isna(row["Symbols"]) or pd.isnull(row["Symbols"]) or pd.isna(row["Title"]) or pd.isnull(row["Title"]):
        continue
    try:
        symbol = row["Symbols"]
        url = "https://cafef.vn/" + row["URL"]
        title = row["Title"]
        raw_content = row["Content"]
        published_at = row["Date"]
        source = "CafeF"

        image_url = banner.loc[banner["URL"] == row["URL"], "imgUrl"].values
        if len(image_url) > 0:
            image_url = image_url[0]
        else:
            image_url = None  # or some default value

        # Check if company exists
        cursor.execute(
            """SELECT id, name, symbol FROM company WHERE symbol = %s""", (symbol,)
        )
        company_record = cursor.fetchone()

        # If company does not exist, skip
        if not company_record:
            continue

        company_id = company_record[0]

        # Check if news exists
        cursor.execute("""SELECT id, title FROM news WHERE title = %s""", (title,))
        news_record = cursor.fetchone()

        # If news exists, skip
        if news_record:
            continue

        news_id, created_at, updated_at = generate_uuid("news")
        # Insert news record
        cursor.execute(
            """INSERT INTO news (id, title, "rawContent", url, banner, "publishedAt", source, "createdAt", "updatedAt", company, "order")
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
            (
                news_id,
                title,
                raw_content,
                url,
                image_url,
                published_at,
                source,
                created_at,
                updated_at,
                company_id,
                0,
            ),
        )
        conn.commit()
        print(f"News: {title} for {symbol}")

    except Exception as e:
        conn.rollback()
        print(f"Error: {e}")
        break
    finally:
        cursor.close()

### Update more news

In [39]:
data = read_data("news/matched_datass_vietstock", raw = True)

In [44]:
data.rename(columns={"0": "Date", "1": "Title", "2": "URL", "3": "Symbol", "4": "Content"}, inplace=True)
data.head()

,Date,Title,URL,Symbol,Content
0,13/06/2024,PTP: Ngày đăng ký cuối cùng trả cổ tức bằng ti...,https://vietstock.vn/2024/06/ptp-ngay-dang-ky-...,PTP,PTP: Ngày đăng ký cuối cùng trả cổ tức bằng ti...
1,07/05/2024,PTP: Bổ nhiệm Ông Phạm Tuấn Anh giữ chức vụ Ph...,https://vietstock.vn/2024/05/ptp-bo-nhiem-ong-...,PTP,PTP: Bổ nhiệm Ông Phạm Tuấn Anh giữ chức vụ Ph...
2,03/05/2024,PTP: Nghị quyết Đại hội đồng cổ đông thường ni...,https://vietstock.vn/2024/05/ptp-nghi-quyet-da...,PTP,PTP: Nghị quyết Đại hội đồng cổ đông thường ni...
3,04/04/2024,PTP: Tài liệu họp Đại hội đồng cổ đông,https://vietstock.vn/2024/04/ptp-tai-lieu-hop-...,PTP,PTP: Tài liệu họp Đại hội đồng cổ đôngTài liệu...
4,25/03/2024,PTP: Báo cáo thường niên 2023,https://vietstock.vn/2024/03/ptp-bao-cao-thuon...,PTP,PTP: Báo cáo thường niên 2023Tài liệu đính kèm...


In [ ]:
def import_news(dataframe):
    company_id = get_company_id()

    prev_close_price = {}
    data = []
    batch_size = 10000

    for _, row in tqdm(dataframe.iterrows(), total=dataframe.shape[0]):
        if pd.isna(row["Symbol"]):
            continue

        symbol = row["Symbol"]

        if symbol not in company_id:
            continue

        id, created_at, updated_at = generate_uuid()

        new_news = News(
            id=id,
            createdAt=created_at,
            updatedAt=updated_at,
            title=row["Title"],
            rawContent=row["Content"],
            cleanContent=row["Content"],
            url=row["URL"],
            banner=None,
            source="Vietstock",
            publishedAt=convert_to_timestamp(row["Date"]),
            order=0,
            company=company_id[symbol],
        )

        data.append(new_news)

        if len(data) % batch_size == 0:
            try:
                session.add_all(data)
                session.commit()
                data = []
            except Exception as e:
                session.rollback()
                print(f"Error: {e}")
                break

        # try:
        #     session.add(new_news)
        #     session.commit()
        # except Exception as e:
        #     session.rollback()
        #     print(f"Error: {e}")
        #     break

## Financial statement

In [20]:
statement = pd.read_csv("./data/cafef/statement.csv")
statement.notnull().sum()

Unnamed: 0    19461
Year          19461
Symbol        19461
EPS           19461
BV            19461
PE            19461
ROA           19461
ROE           19461
ROS           19461
GOS           19461
DAR           19461
dtype: int64

In [ ]:
for index, row in tqdm(statement.iterrows(), total=statement.shape[0]):
    cursor = conn.cursor()
    if (
        pd.isna(row["Symbol"])
        or pd.isnull(row["Symbol"])
        or pd.isna(row["Year"])
        or pd.isnull(row["Year"])
    ):
        continue
    try:
        symbol = row["Symbol"]
        year = row["Year"].__str__()
        pe = row["PE"]
        roa = row["ROA"]
        roe = row["ROE"]
        eps = row["EPS"]
        bvps = row["BV"]
        ros = row["ROS"]
        dar = row["DAR"]
        gos = row["GOS"]

        # Check if company exists
        cursor.execute(
            """SELECT id, name, symbol FROM company WHERE symbol = %s""", (symbol,)
        )
        company_record = cursor.fetchone()

        # If company does not exist, skip
        if not company_record:
            continue

        company_id = company_record[0]

        # Check if statement exists
        cursor.execute(
            """SELECT id, company, year from financialstatement WHERE company = %s AND year = %s""",
            (company_id, year),
        )

        statement_record = cursor.fetchone()

        # If statement exists, skip
        if statement_record:
            continue

        statement_id, created_at, updated_at = generate_uuid("statement")
        # Insert statement record
        cursor.execute(
            """INSERT INTO financialstatement (id, year, "PE", "ROA", "ROE", "EPS", "BVPS", "ROS", "DAR", "GOS", "createdAt", "updatedAt", company, "order")
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
            (
                statement_id,
                year,
                pe,
                roa,
                roe,

                
                eps,
                bvps,
                ros,
                dar,
                gos,
                created_at,
                updated_at,
                company_id,
                0,
            ),
        )
        conn.commit()
        print(f"Statement: {year} for {symbol}")
        logging.info(f"Statement: {year} for {symbol}")

    except Exception as e:
        conn.rollback()
        print(f"Error: {e}")
        break
    finally:
        cursor.close()

100%|██████████| 19461/19461 [01:05<00:00, 295.33it/s]


In [18]:
def get_statement():
    results = session.query(Statement.year, Statement.company).all()
    statement = [(result[0], result[1]) for result in results]
    return statement

In [19]:
statement = get_statement()

In [20]:
data = read_data("statement/financialstatement", raw=True)

# get duplicate rows by year and company
duplicate_rows = data[data.duplicated(subset=["company", "year"], keep=False)]

# drop where GOS is equal to 2000
duplicate_rows = duplicate_rows[duplicate_rows["GOS"] != 2000]

# get unique rows by year and company
unique_rows = data.drop_duplicates(subset=["company", "year"], keep=False)
len(duplicate_rows)

39

In [21]:
def get_company_id():
    results = session.query(Company.id).all()
    company_id = [result[0] for result in results]
    results = session.query(Statement.company).all()
    statement_company_id = [result[0] for result in results]
    return company_id, statement_company_id

In [22]:
company_id, statement_company_id = get_company_id()

In [34]:
for id in statement_company_id:
    if id not in company_id:
        print(id)

data_company = data["company"].unique().tolist()
count = 0
for id in company_id:
    if id not in data_company:
        print(id)
        count += 1

print(count)

0b14819c-f5fe-4f3d-ac1f-3c3ca94b3325
0ce8c2b1-6e15-44fe-b80f-710fd9ff9b07
3152f439-f5c1-4c2a-8f16-c6247a8274a8
32e2ed52-967f-4ab2-9eb5-243f85f96bde
4bd0eea8-9343-402b-96b5-277f2cbb603c
62f56980-8aaa-4e90-b126-7d49e4b2863c
6c830f38-5d7b-4a5f-a477-7ae8345e4910
7057f357-cda9-48e0-9acd-3d703ec5018f
711d6452-0724-4307-ae8b-bd674e8f0c75
7d4a9cf8-3f12-48fa-989f-5f438dabf966
885b67f9-f51f-439b-8c25-d13a5ffae346
925e7503-0da4-4e19-afde-2ba0dcc62e07
9a96aa38-9c6f-4375-a154-e369015f5b34
b11b01c0-5779-4e4e-a184-49b08675ebf8
b250e316-eb77-4279-b57b-6879e950992a
b7b1df08-2fb7-4e32-b4bd-0f28d6ae1339
c5fda156-d85d-488a-a12b-7d203bc48c47
da47541a-6561-437a-b4f9-1a1ae989506b
f91a3bc9-1af1-4d3d-8130-25745b96f9f6
19


In [35]:
count = 0
for _, row in data.iterrows():
    company = row["company"]
    year = row["year"].__str__()

    if (year, company) not in statement:
        count += 1
    else:
        continue
        # print(f"Company: {company}, Year: {year}")

print(count)

1615


## Stock


In [14]:
stock = pd.read_csv(f"{data_path}/cafef/stock.csv")
stock.notnull().sum()

Symbol            1774368
Date              1774368
Adjusted Price    1774368
Closing Price     1774368
Opening Price     1774368
Lowest Price      1774368
Highest Price     1774368
Change            1774368
Traded Volume     1774368
Traded Value      1774368
Agreed Volume     1774368
Agreed Value      1774368
dtype: int64

In [8]:
stored_company_id = None
stored_symbol = None

In [32]:
company_query = "SELECT id, symbol, exchange FROM company"
company_info = pd.read_sql(company_query, engine)

# Create the dictionary
company_dict = {
    row['symbol']: {'id': row['id'], 'exchange': row['exchange']}
    for _, row in company_info.iterrows()
}

# Print the resulting dictionary
print(company_dict)

{'ADP': {'id': '5586e757-3de6-41b8-949c-9609afa3307c', 'exchange': 'HOSE'}, 'PTP': {'id': '0aecbefd-de78-4735-b426-5bc09f7f65f5', 'exchange': 'UPCOM'}, 'ST8': {'id': 'ad249263-df58-4739-8178-b9fe63d21e6e', 'exchange': 'HOSE'}, 'BVH': {'id': '0473594f-6d2f-4e30-a2d8-54e0091c685c', 'exchange': 'HOSE'}, 'PVR': {'id': 'e5d8dc57-117a-4e0c-8eb5-a2f09c78873c', 'exchange': 'UPCOM'}, 'NCS': {'id': 'dba2c82d-a257-48ec-ab0a-88b21b6e59f0', 'exchange': 'UPCOM'}, 'TQN': {'id': '9341224a-939e-44d1-b9a0-1e5fc4ea3724', 'exchange': 'UPCOM'}, 'APS': {'id': '8ad6da00-1632-4420-ae22-39a2714d81d2', 'exchange': 'HNX'}, 'BTH': {'id': '89bfe003-85ba-4310-8b12-91d0d362bd0a', 'exchange': 'UPCOM'}, 'AAH': {'id': 'c680324e-d37d-4eec-92f4-3a2b61de1469', 'exchange': 'UPCOM'}, 'SCG': {'id': 'b9be5669-a77e-4ff6-b018-ba32a327e08d', 'exchange': 'HNX'}, 'SBT': {'id': 'd24b3873-3e57-4bac-929f-af05d7809794', 'exchange': 'HOSE'}, 'HLY': {'id': '899b312c-607f-4e07-a408-f5eca3e07a5a', 'exchange': 'UPCOM'}, 'DS3': {'id': 'fdbc

In [15]:
data = []
prev_close_price = {}
batch_size = 1000
insert_query = text("""
INSERT INTO stock (id, "date", "openingPrice", "closingPrice", "highestPrice", "lowestPrice", "ceilingPrice", "floorPrice", "adjustedPrice", change, "tradedVolume", "tradedValue", "agreedVolume", "agreedValue", "createdAt", "updatedAt", company, "order")
VALUES (:id, :date, :openingPrice, :closingPrice, :highestPrice, :lowestPrice, :ceilingPrice, :floorPrice, :adjustedPrice, :change, :tradedVolume, :tradedValue, :agreedVolume, :agreedValue, :createdAt, :updatedAt, :company, :order)
""")

count = 0
for index, row in tqdm(stock.iterrows(), total=stock.shape[0]):
    if pd.isna(row["Symbol"]) or pd.isna(row["Closing Price"]):
        continue

    symbol = row["Symbol"]
    close_price = row["Closing Price"]
    open_price = row["Opening Price"]
    highest_price = row["Highest Price"]
    lowest_price = row["Lowest Price"]
    change_price = row["Adjusted Price"]
    change = row["Change"]
    traded_volume = row["Traded Volume"]
    traded_value = row["Traded Value"]
    agreed_volume = row["Agreed Volume"]
    agreed_value = row["Agreed Value"]
    date = convert_to_timestamp(row["Date"])

    # Lấy exchange từ dictionary
    company_info = company_dict.get(symbol, {})
    exchange = company_info.get("exchange")
    company_id = company_info.get("id")

    # Lấy giá đóng cửa của ngày trước để tính giá trần và giá sàn
    if symbol in prev_close_price:
        ceiling_price, floor_price = calculate_ceiling_floor_price(
            exchange, prev_close_price[symbol]
        )
    else:
        ceiling_price, floor_price = 0, 0

    prev_close_price[symbol] = close_price

    # Tạo id, created_at, updated_at
    stock_id, created_at, updated_at = generate_uuid("stock")

    # Thêm dữ liệu vào danh sách
    data.append(
        {
            "id": stock_id,
            "date": date,
            "openingPrice": open_price,
            "closingPrice": close_price,
            "highestPrice": highest_price,
            "lowestPrice": lowest_price,
            "ceilingPrice": ceiling_price,
            "floorPrice": floor_price,
            "adjustedPrice": change_price,
            "change": change,
            "tradedVolume": traded_volume,
            "tradedValue": traded_value,
            "agreedVolume": agreed_volume,
            "agreedValue": agreed_value,
            "createdAt": created_at,
            "updatedAt": updated_at,
            "company": company_id,
            "order": 0,
        }
    )
    count += 1

    if count % batch_size == 0:
        with engine.connect() as conn:
            conn.execute(insert_query, data)
            conn.commit()
            data = []

if data:
    try:
        with engine.connect() as conn:
            conn.execute(insert_query, data)
            conn.commit()
            print("Remaining records inserted.")
    except Exception as e:
        print(f"Error occurred: {e}")

print("Data insertion complete.")


100%|██████████| 1774368/1774368 [08:56<00:00, 3305.19it/s]

Remaining records inserted.
Data insertion complete.


In [18]:
data = read_data("stock/processed_prices", raw=True)
# sort data by company, date
data = data.sort_values(by=["symbols", "Date"])
data = data.reset_index(drop=True)
data.keys().tolist()

['symbols',
 'Date',
 'open',
 'close',
 'adjust',
 'high',
 'low',
 'change',
 'GiaTriKhopLenh',
 'KhoiLuongKhopLenh',
 'GiaTriThoaThuan',
 'KhoiLuongThoaThuan',
 'id',
 'company',
 'order',
 'createdAt',
 'updatedAt']

In [38]:
def import_stock(dataframe):
    # company_id = get_company_id()

    prev_close_price = {}
    data = []
    batch_size = 10000

    for _, row in tqdm(dataframe.iterrows(), total=dataframe.shape[0]):
        if pd.isna(row["symbols"]) or pd.isna(row["close"]):
            continue

        symbol = row["symbols"]
        # company = company_id.get(symbol)
        # if company is None:
        #     continue

        # if company != row["company"]:
        #     print(f"{symbol} - {company} - {row['company']}")
        #     break

        close_price = row["close"]

        company_info = company_dict.get(symbol, {})
        exchange = company_info.get("exchange")
        company_id = company_info.get("id")

        if symbol in prev_close_price:
            ceiling_price, floor_price = calculate_ceiling_floor_price(
                exchange, prev_close_price[symbol]
            )
        else:
            ceiling_price, floor_price = 0, 0

        prev_close_price[symbol] = close_price

        id, created_at, updated_at = generate_uuid()

        date = convert_to_timestamp(row["Date"])

        new_stock = Stock(
            id = id,
            createdAt = created_at,
            updatedAt = updated_at,
            openingPrice = round(row["open"], 2),
            closingPrice = round(close_price, 2),
            ceilingPrice = round(ceiling_price, 2),
            floorPrice = round(floor_price, 2),
            highestPrice = round(row["high"], 2),
            lowestPrice = round(row["low"], 2),
            adjustedPrice = round(row["adjust"], 2),
            change = row["change"],
            tradedValue = row["GiaTriKhopLenh"],
            tradedVolume = row["KhoiLuongKhopLenh"],
            agreedValue = row["GiaTriThoaThuan"],
            agreedVolume = row["KhoiLuongThoaThuan"],
            date = date,
            order = 0,
            company = company_id,
        )

        data.append(new_stock)

        if len(data) % batch_size == 0:
            try:
                session.add_all(data)
                session.commit()
                data = []
            except Exception as e:
                session.rollback()
                print(f"Error: {e}")
                break

        # try:
        #     session.add(new_stock)
        #     session.commit()
        # except Exception as e:
        #     session.rollback()
        #     print(f"Error: {e}")
        #     break

In [40]:
try:
    import_stock(data)
except Exception as e:
    session.rollback()
    print(f"Error during import: {e}")

100%|██████████| 3829386/3829386 [40:58<00:00, 1557.78it/s] 


## Report

In [36]:
data = read_data("report/report_quarter", raw = True)
data.keys().tolist()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9720\2404672310.py:2: DtypeWarning: Columns (18,19,21,22,23,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f"{data_path}/raw/{filename}.csv") if raw else pd.read_csv(f"{data_path}/beautiful/{filename}.csv")


['Mã',
 'Quý',
 'Tổng nợ',
 'Nợ ngắn hạn',
 'Tổng tài sản',
 'Lợi nhuận khác',
 'Giá vốn hàng bán',
 'Vốn chủ sở hữu',
 'Lợi nhuận sau thuế',
 'Lợi nhuận tài chính',
 'Doanh thu bán hàng và CCDV',
 'Lợi nhuận gộp về BH và CCDV',
 'Tổng lợi nhuận trước thuế',
 'Tổng tài sản lưu động ngắn hạn',
 'Lợi nhuận sau thuế của công ty mẹ',
 'Tài sản ngắn hạn',
 'Tiền',
 'Các khoảng tương đương tiền',
 'Đầu tư công ty con',
 'Nợ phải trả',
 'Vay dài hạn',
 'Vay ngắn hạn',
 'Đầu tư ngắn hạn',
 'Đầu tư dài hạn',
 'Cổ phiếu ưu đãi',
 'Nợ phải thu khách hàng',
 'Hàng tồn kho',
 'Tài sản cố định hữu hình',
 'M-score',
 'M-score**',
 'F-score',
 "Z-score'",
 'Z-score"',
 'Z-score']

### Import reports

In [39]:
def import_report(dataframe, month=False):
    company_id = get_company_id()
    for _, row in tqdm(dataframe.iterrows(), total=dataframe.shape[0]):
        detail = {
            "Sales revenue": row["Doanh thu bán hàng và CCDV"],
            "Cost of goods sold": row["Giá vốn hàng bán"],
            "Gross profit from sales and services": row["Lợi nhuận gộp về BH và CCDV"],
            "Financial profit": row["Lợi nhuận tài chính"],
            "Other profit": row["Lợi nhuận khác"],
            "Total profit before tax": row["Tổng lợi nhuận trước thuế"],
            "Profit after tax": row["Lợi nhuận sau thuế"],
            "Profit after tax of parent company": row["Lợi nhuận sau thuế của công ty mẹ"],
            "Total current assets": row["Tổng tài sản lưu động ngắn hạn"],
            "Total assets": row["Tổng tài sản"],
            "Short-term liabilities": row["Nợ ngắn hạn"],
            "Total liabilities": row["Tổng nợ"],
            "Owner's equity": row["Vốn chủ sở hữu"],
            "Current assets": row["Tài sản ngắn hạn"],
            "Cash": row["Tiền"],
            "Cash equivalents": row["Các khoảng tương đương tiền"],
            "Investment in subsidiaries": row["Đầu tư công ty con"],
            "Liabilities": row["Nợ phải trả"],
            "Long-term loans": row["Vay dài hạn"],
            "Short-term loans": row["Vay ngắn hạn"],
            "Short-term investments": row["Đầu tư ngắn hạn"],
            "Long-term investments": row["Đầu tư dài hạn"],
            "Preferred stock": row["Cổ phiếu ưu đãi"],
            "Receivables from customers": row["Nợ phải thu khách hàng"],
            "Inventory": row["Hàng tồn kho"],
            "Tangible fixed assets": row["Tài sản cố định hữu hình"],
        }

        quarter = row["Quý"]
        if (quarter is not None) and (quarter is not pd.NA):
            quarter, year = quarter.split("-")
            quarter = int(quarter[-1])
        else:
            quarter = 0
            year = row["Năm"]

        id, created_at, updated_at = generate_uuid()

        if row["Mã"] not in company_id:
            continue

        new_report = Report(
            year=year,
            quarter=quarter,
            month=month,
            detail=json.dumps(detail, ensure_ascii=False),
            mScore1=round(row["M-score"], 2),
            mScore2=round(row["M-score**"], 2),
            fScore1=round(row["F-score"], 2),
            zScore1=round(row["Z-score"], 2),
            zScore2=round(row["Z-score'"], 2),
            zScore3=round(row['Z-score"'], 2),
            company=company_id[row["Mã"]],
            id=id,
            createdAt=created_at,
            updatedAt=updated_at,
            order=0,
        )
        
        try:
            session.add(new_report)
            session.commit()
        except Exception as e:
            # session.rollback()
            print(f"Error: {e} - new_report: {new_report}")
            break

In [40]:
try:
    import_report(data)
except Exception as e:
    session.rollback()
    print(f"Error during import_report: {e}")

  0%|          | 0/64180 [00:00<?, ?it/s]

100%|██████████| 64180/64180 [02:24<00:00, 445.11it/s]
